In [38]:
import pandas as pd
import numpy as np

requests = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/2cb3f1e5-7961-4034-9643-0adb38a1fefa/csv")
assessments = pd.read_csv("https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/96aff67b-a51f-4982-97ce-b720006edf71/csv")


In [39]:
requests = requests[requests["breakdown_topic"]=="All requests for EHC needs assessments"]
requests = requests[requests["old_la_code"].notna()]
#requests_region = requests[["time_period","region_name","requests_received_in_year"]]
requests_la = requests[["time_period","old_la_code","requests_received_in_year","requests_decided_not_to_assess"]]
requests_la["requests_received_in_year"]=pd.to_numeric(requests_la["requests_received_in_year"],errors="coerce")
requests_la["requests_decided_not_to_assess"]=pd.to_numeric(requests_la["requests_decided_not_to_assess"],errors="coerce")
requests_la["request_not_to_assess_pc"]=requests_la["requests_decided_not_to_assess"]/requests_la["requests_received_in_year"]*100

assessments = assessments[assessments['breakdown_topic']=="All EHC needs assessments"]
assessments = assessments[assessments["old_la_code"].notna()]
assessments_la=assessments[["time_period","old_la_code","assess_not_issued_pc"]]

requests_la=requests_la.merge(assessments_la,on=["old_la_code","time_period"],how="left")

requests_la["assess_not_issued_pc"] = (requests_la["assess_not_issued_pc"].replace("x", np.nan).astype(float))

#requests_la.head(5)
#assessments_la.head(10)

In [40]:
#Option 1 
#Calculating a crude-average (YoY increase) forecast of requests per LA given SEN2 published data.
requests_la=requests_la.sort_values(["old_la_code","time_period"])
requests_la["requests_ffill"] = requests_la.groupby("old_la_code")["requests_received_in_year"].ffill()
#the above line is needed to deal with missing years of data requests
requests_la["pct_change"]=requests_la.groupby("old_la_code")["requests_ffill"].pct_change()*100
avg_pct_req_la_crude = (requests_la.groupby("old_la_code")["pct_change"].mean(skipna=True).reset_index().rename(columns={"pct_change":"avg_pct_change"}))

#Calculating a crude-average forecast for % of no to assess and no to issue. 
avg_R2A = (requests_la.groupby("old_la_code")["request_not_to_assess_pc"].mean(skipna=True).reset_index().rename(columns={"request_not_to_assess_pc":"avg_R2A_pct"}))
avg_R2I = (requests_la.groupby("old_la_code")["assess_not_issued_pc"].mean(skipna=True).reset_index().rename(columns={"assess_not_issued_pc":"avg_R2I_pct"}))

avg_pct_req_la_crude = (avg_pct_req_la_crude.merge(avg_R2A, on="old_la_code", how="left").merge(avg_R2I, on="old_la_code", how="left"))

#avg_pct_req_la_crude.head(5)
#requests_la.head(10)

In [41]:
#Save output to Excel file to use in Excel Tool.
import sys
!{sys.executable} -m pip install XlsxWriter


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [42]:
#Option 2
#Calculating a most recent year forecast, based on the most recent SEN2 request data and most recent % difference behaviour, most recent % of no to assess and no to plan.
requests_la=requests_la.sort_values(["old_la_code","time_period"])
latest_pct=requests_la.groupby("old_la_code").tail(1)[["old_la_code","time_period","pct_change","request_not_to_assess_pc","assess_not_issued_pc"]]

In [43]:
#Option 3
#Weighted Average (last 3 years only)
weights = np.array([1,2,3])

#Define a new function for weighted average calculation. 
def weighted_avg_last3(series):
    vals = series.dropna()
    if len(vals)==0:
        return np.nan
    last_vals=vals.tail(3).values

    w=weights[-len(last_vals):]

    return np.average(last_vals,weights=w)

weighted_pct_avg_diff_la=(requests_la.groupby("old_la_code")["pct_change"].apply(weighted_avg_last3).reset_index(name="weighted_pct_change"))

r2a_weighted = (requests_la.groupby("old_la_code")["request_not_to_assess_pc"].apply(weighted_avg_last3).reset_index(name="weighted_pct_R2A"))

r2i_weighted = (requests_la.groupby("old_la_code")["assess_not_issued_pc"].apply(weighted_avg_last3).reset_index(name="weighted_pct_R2I"))   


weighted_pct_avg_diff_la = (weighted_pct_avg_diff_la.merge(r2a_weighted, on="old_la_code", how="left").merge(r2i_weighted, on="old_la_code", how="left"))

weighted_pct_avg_diff_la.head(5)


,old_la_code,weighted_pct_change,weighted_pct_R2A,weighted_pct_R2I
0,201.0,214.393939,1.515152,0.000000
1,202.0,15.037363,7.551245,4.416667
2,203.0,25.639026,12.454628,8.366667
3,204.0,10.141710,28.022692,0.240000
4,205.0,5.955130,31.422186,10.633333


In [44]:
requests_la.head(10)

,time_period,old_la_code,requests_received_in_year,requests_decided_not_to_assess,request_not_to_assess_pc,assess_not_issued_pc,requests_ffill,pct_change
84,2019,201.0,8.0,3.0,37.500000,NaN,8.0,NaN
235,2020,201.0,5.0,1.0,20.000000,NaN,5.0,-37.500000
387,2021,201.0,1.0,1.0,100.000000,NaN,1.0,-80.000000
539,2022,201.0,11.0,1.0,9.090909,0.0,11.0,1000.000000
692,2023,201.0,2.0,0.0,0.000000,0.0,2.0,-81.818182
845,2024,201.0,5.0,0.0,0.000000,0.0,5.0,150.000000
85,2019,202.0,230.0,0.0,0.000000,NaN,230.0,NaN
236,2020,202.0,174.0,0.0,0.000000,NaN,174.0,-24.347826
388,2021,202.0,140.0,31.0,22.142857,NaN,140.0,-19.540230
540,2022,202.0,203.0,30.0,14.778325,9.7,203.0,45.000000


In [45]:
#Option 4
#Linear projection (trend based on historic YoY% change and Refuse to Assess and Issue %)
#Taming ouliers first (per LA)

def tame_outliers(group: pd.DataFrame, value_col: str, out_col: str,
                  k: float = 2.5, global_clip=(0, 100)) -> pd.DataFrame:
    g = group.copy()
    y = g[value_col]
    
    if y.notna().sum() < 3:
        g[out_col] = y.clip(*global_clip)
        g[f"{out_col}_is_outlier"] = False
        return g
    #calculate median and MAD (Median Absolute Deviation) for a robust z-score. 
    #using MAD works better than standard deviation on short, noisy series.
    med = y.median(skipna=True)
    mad = (y - med).abs().median(skipna=True)
    
    if mad == 0 or np.isnan(mad):
        lower, upper = global_clip
    else:
        scale = 1.4826 * mad
        lower = max(med - k * scale, global_clip[0])
        upper = min(med + k * scale, global_clip[1])

    g[out_col] = y.clip(lower, upper)
    g[f"{out_col}_is_outlier"] = (y < lower) | (y > upper)
    return g

#Tame YoY%



tamed_pct = (
    requests_la.groupby("old_la_code", group_keys=True)
    .apply(lambda g: tame_outliers(
        g,
        value_col="pct_change",
        out_col="pct_tamed",
        k=2.5,
        global_clip=(-100, 200)
    ))
    .reset_index(level=0)   # <-- CRITICAL FIX
    [["old_la_code","time_period","pct_tamed"]]
)


#Tame Refuse to Assess and Refuse to Issue % 
tamed_r2a = (
    requests_la.groupby("old_la_code", group_keys=True).apply(lambda g: tame_outliers(g, value_col="request_not_to_assess_pc",
                                   out_col="r2a_tamed",
                                   k=2.5, global_clip=(0, 100))).reset_index(level=0)[["old_la_code","time_period","r2a_tamed"]])


tamed_r2i = (
    requests_la.groupby("old_la_code", group_keys=True)
    .apply(lambda g: tame_outliers(
        g,
        value_col="assess_not_issued_pc",
        out_col="r2i_tamed",
        k=2.5,
        global_clip=(0, 100)
    ))
    .reset_index(level=0)   # <-- CRITICAL FIX
    [["old_la_code","time_period","r2i_tamed"]]
)



requests_tamed = (
    requests_la
    .merge(tamed_pct, on=["old_la_code","time_period"], how="left")
    .merge(tamed_r2a, on=["old_la_code","time_period"], how="left")
    .merge(tamed_r2i, on=["old_la_code","time_period"], how="left")
)



#Fits a linear regression for YoY%

def linear_trend_by_column(df: pd.DataFrame,
                           value_col: str,
                           group_col: str = "old_la_code",
                           time_col: str = "time_period",
                           latest_aux_cols=("requests_received_in_year",)) -> pd.DataFrame:
    
    df = df.sort_values([group_col, time_col]).copy()
    results = []

    for la, g in df.groupby(group_col):
        sub = g.dropna(subset=[value_col])
        latest_year = g[time_col].max()
        latest_vals = {}
        for c in latest_aux_cols:
            latest_vals[f"latest_{c}"] = (
                g[c].dropna().iloc[-1] if c in g.columns and g[c].notna().any() else np.nan)
            
        if sub.shape[0] < 2:
            results.append({
                group_col: la,
                "latest_value": sub[value_col].dropna().iloc[-1] if sub[value_col].notna().any() else np.nan,
                "linear_slope_per_year": np.nan,
                "linear_intercept": np.nan,
                "linear_proj_next_year": np.nan,
                **latest_vals})
            continue
        
        x = sub[time_col].astype(float).values
        y = sub[value_col].astype(float).values
        slope, intercept = np.polyfit(x, y, 1)
        next_year = float(latest_year) + 1.0
        proj_next = slope * next_year + intercept
        results.append({
            group_col: la,
            "latest_value": sub[value_col].dropna().iloc[-1],
            "linear_slope_per_year": slope,
            "linear_intercept": intercept,
            "linear_proj_next_year": proj_next,
            **latest_vals})
    return pd.DataFrame(results)

#Apply formula to YoY % change forecast

trend_yoy = linear_trend_by_column(
    requests_tamed, value_col="pct_tamed",
    latest_aux_cols=("requests_received_in_year",)
)

# Forecast Refuse to Assess
trend_r2a = linear_trend_by_column(
    requests_tamed, value_col="r2a_tamed",
    latest_aux_cols=()
)
trend_r2a = trend_r2a.rename(columns={
    "latest_value": "r2a_latest",
    "linear_slope_per_year": "r2a_slope",
    "linear_intercept": "r2a_intercept",
    "linear_proj_next_year": "r2a_linear_proj"
})
trend_r2a["r2a_linear_proj"] = trend_r2a["r2a_linear_proj"].clip(0, 100)

# Forecast Refuse to Issue
trend_r2i = linear_trend_by_column(
    requests_tamed, value_col="r2i_tamed",
    latest_aux_cols=()
)
trend_r2i = trend_r2i.rename(columns={
    "latest_value": "r2i_latest",
    "linear_slope_per_year": "r2i_slope",
    "linear_intercept": "r2i_intercept",
    "linear_proj_next_year": "r2i_linear_proj"
})
trend_r2i["r2i_linear_proj"] = trend_r2i["r2i_linear_proj"].clip(0, 100)

#merge into one single data frame
trend_df = (
    trend_yoy
    .merge(trend_r2a, on="old_la_code", how="left")
    .merge(trend_r2i, on="old_la_code", how="left")
)

print(trend_df.columns.tolist())


['old_la_code', 'latest_value', 'linear_slope_per_year', 'linear_intercept', 'linear_proj_next_year', 'latest_requests_received_in_year', 'r2a_latest', 'r2a_slope', 'r2a_intercept', 'r2a_linear_proj', 'r2i_latest', 'r2i_slope', 'r2i_intercept', 'r2i_linear_proj']


In [46]:
#Suggestion to LA regarding which forecasting model to use based on their LA data behaviour. 
#Create master table with all 4 forecasting method for each LA:

forecast_all=avg_pct_req_la_crude.copy()
forecast_all=forecast_all.merge(latest_pct, on="old_la_code",how="left")
forecast_all = forecast_all.merge(weighted_pct_avg_diff_la, on="old_la_code", how="left")
forecast_all = forecast_all.merge(trend_df, on="old_la_code", how="left")

print(forecast_all.columns.tolist())

#adding behaviour metrics
# Volatility proxy across available summary %s
summary_cols = ["avg_pct_change", "weighted_pct_change", "pct_change"]
def row_std_proxy(row):
    vals = [row[c] for c in summary_cols if pd.notna(row[c])]
    return float(np.nanstd(vals, ddof=0)) if len(vals) >= 2 else np.nan

forecast_all["volatility_proxy"] = forecast_all.apply(row_std_proxy, axis=1)

# Recent shock measure: how different last year is vs the 3yr weighted
forecast_all["recent_shock_abs"] = (
    (forecast_all["pct_change"] - forecast_all["weighted_pct_change"]).abs()
)

# Trend strength (already have slope)
forecast_all["trend_strength"] = forecast_all["linear_slope_per_year"].abs()

# If you truly cannot bring missing years from source, set to 0 (or leave NaN)
forecast_all["missing_years"] = 0  # Placeholder if you can’t compute from requests_la

#Model Selection Logic

def recommend_model_row(row):
    vol = row["volatility_proxy"]
    slope = row["linear_slope_per_year"]
    missing = row["missing_years"]
    last_yoy = row["pct_change"]
    weighted3 = row["weighted_pct_change"]

    # ---------- OPTION 4: Linear Trend ----------
    # High volatility OR strong trend OR many missing years
    if (not pd.isna(vol) and vol > 30) or \
       (not pd.isna(slope) and abs(slope) > 5) or \
       (missing >= 2):
        return "Option 4: Linear Trend"

    # ---------- OPTION 3: Weighted Last 3 ----------
    # Stable behaviour
    if (not pd.isna(vol) and vol <= 10) and missing == 0:
        # but check if last-year shock is large → then Option 2
        if pd.notna(last_yoy) and pd.notna(weighted3) and abs(last_yoy - weighted3) > 25:
            return "Option 2: Last-Year Only"
        return "Option 3: Weighted Last 3"

    # ---------- Moderate Behaviour ----------
    if (not pd.isna(vol)) and (10 < vol <= 30):
        if not pd.isna(slope) and abs(slope) > 3:
            return "Option 4: Linear Trend"
        else:
            return "Option 3: Weighted Last 3"

    # ---------- OPTION 1: fallback ----------
    return "Option 1: Crude Average"


forecast_all["recommended_method"] = forecast_all.apply(recommend_model_row, axis=1)

#Chose correct model applied for each LA data behaviour.

def choose_pct(row):
    m = row["recommended_method"]
    if m == "Option 1: Crude Average":
        return row["avg_pct_change"]
    if m == "Option 2: Last-Year Only":
        return row["pct_change"]
    if m == "Option 3: Weighted Last 3":
        return row["weighted_pct_change"]
    if m == "Option 4: Linear Trend":
        return row["linear_proj_next_year"]
    return np.nan

forecast_all["chosen_pct_change"] = forecast_all.apply(choose_pct, axis=1)

forecast_all=forecast_all[["old_la_code","volatility_proxy", "recent_shock_abs", "trend_strength", "missing_years", "recommended_method", "chosen_pct_change"]]






['old_la_code', 'avg_pct_change', 'avg_R2A_pct', 'avg_R2I_pct', 'time_period', 'pct_change', 'request_not_to_assess_pc', 'assess_not_issued_pc', 'weighted_pct_change', 'weighted_pct_R2A', 'weighted_pct_R2I', 'latest_value', 'linear_slope_per_year', 'linear_intercept', 'linear_proj_next_year', 'latest_requests_received_in_year', 'r2a_latest', 'r2a_slope', 'r2a_intercept', 'r2a_linear_proj', 'r2i_latest', 'r2i_slope', 'r2i_intercept', 'r2i_linear_proj']


/usr/local/python/3.12.1/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1845: RuntimeWarning: invalid value encountered in subtract
  np.subtract(arr, avg, out=arr, casting='unsafe', where=where)


In [47]:
with pd.ExcelWriter("SEND Performance and Capacity Tool - Calculation.xlsx", engine="xlsxwriter") as writer:
    avg_pct_req_la_crude.to_excel(writer, sheet_name="Crude Average", index=False)
    latest_pct.to_excel(writer, sheet_name="Latest Average", index=False)
    weighted_pct_avg_diff_la.to_excel(writer,sheet_name="Weighted Average",index=False)
    trend_df.to_excel(writer,sheet_name="Linear Trend",index=False)
    forecast_all.to_excel(writer,sheet_name="Suggested Forecast",index=False)